# 2. Conception, Entraînement, Évaluation et Déploiement du Modèle CNN

Ce notebook couvre :
- L'architecture CNN (Conv2D + MaxPooling + Dropout + Dense)
- Le choix des activations
- La compilation (Adam + categorical_crossentropy)
- model.summary() et plot_model()
- Les hyperparamètres et la mesure du temps
- L'entraînement avec ModelCheckpoint
- L'évaluation, les courbes d'apprentissage
- Matrice de confusion + rapport de classification
- Exemples corrects/incorrects
- Sauvegarde du modèle (.h5/.keras)
- Fonction predict_image(path)
- Interface Streamlit (src/app.py)

In [1]:
import os
import time
import warnings

import cv2
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support
)

import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Dropout,
    Flatten,
    Dense,
    Input
)
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import plot_model

warnings.filterwarnings('ignore')

CATEGORIES = ['glioma', 'meningioma', 'notumor', 'pituitary']
IMG_SIZE = 224
RANDOM_STATE = 42

# Répertoires
PROJECT_DIR = os.getcwd()
while not all(os.path.isdir(os.path.join(PROJECT_DIR, 'data', c)) for c in CATEGORIES):
    parent = os.path.dirname(PROJECT_DIR)
    if parent == PROJECT_DIR:
        pass
    PROJECT_DIR = parent

MODELS_DIR = os.path.join(PROJECT_DIR, 'models')
SRC_DIR = os.path.join(PROJECT_DIR, 'src')
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(SRC_DIR, exist_ok=True)

print('TF version:', tf.__version__)
print('PROJECT_DIR:', PROJECT_DIR)
print('MODELS_DIR:', MODELS_DIR)
print('SRC_DIR:', SRC_DIR)

I0000 00:00:1792144836.823406  202213 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792144837.357652  202213 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792144841.491713  202213 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TF version: 2.21.0
PROJECT_DIR: /home/abdo/Documents/Brain-IRM-CNN
MODELS_DIR: /home/abdo/Documents/Brain-IRM-CNN/models
SRC_DIR: /home/abdo/Documents/Brain-IRM-CNN/src


## 2.1 Chargement des données prétraitées

On reprend les données préparées dans `pretraitement.ipynb` : X_train, X_test, y_train, y_test, ainsi que class_weight_dict.

In [2]:
# Recharger rapidement les données (réutilisable)
import os, random, cv2, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.utils import to_categorical

CATEGORIES = ['glioma', 'meningioma', 'notumor', 'pituitary']
IMG_SIZE = 224
VALID_EXTENSIONS = ('.jpeg', '.jpg', '.bmp', '.png')
RANDOM_STATE = 42

PROJECT_DIR = os.getcwd()
while not all(os.path.isdir(os.path.join(PROJECT_DIR, 'data', c)) for c in CATEGORIES):
    parent = os.path.dirname(PROJECT_DIR)
    if parent == PROJECT_DIR:
        raise FileNotFoundError
    PROJECT_DIR = parent
DATA_DIR = os.path.join(PROJECT_DIR, 'data')

# Charger données
images, labels = [], []
for category in CATEGORIES:
    path = os.path.join(DATA_DIR, category)
    class_num = CATEGORIES.index(category)
    for img_name in os.listdir(path):
        if not img_name.lower().endswith(VALID_EXTENSIONS):
            continue
        try:
            img = cv2.imread(os.path.join(path, img_name))
            if img is None: continue
            img_r = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            images.append(img_r); labels.append(class_num)
        except:
            pass

pairs = list(zip(images, labels)); random.shuffle(pairs)
images, labels = zip(*pairs)
X = np.array(images, dtype=np.float32); y = np.array(labels)
y_cat = to_categorical(y, len(CATEGORIES))
X_train, X_test, y_train, y_test = train_test_split(
    X, y_cat, test_size=0.2, random_state=RANDOM_STATE, stratify=y_cat
)
X_train, X_test = X_train/255.0, X_test/255.0
class_weights = compute_class_weight('balanced', classes=np.arange(len(CATEGORIES)), y=y)
class_weight_dict = {i: float(class_weights[i]) for i in range(len(CATEGORIES))}
print('Données prêtes:', X_train.shape, X_test.shape, y_train.shape, y_test.shape)

Données prêtes: (5618, 224, 224, 3) (1405, 224, 224, 3) (5618, 4) (1405, 4)


## 2.2 Architecture CNN

Conv2D + MaxPooling + Dropout + Dense. 
- Couches cachées : ReLU
- Couche de sortie : Softmax (4 classes)

In [ ]:
# Définition du modèle CNN
model = Sequential([
    Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(256, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.3),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(len(CATEGORIES), activation='softmax')
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 24, 24, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 36864)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │    18,874,880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │         1,028 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,395,652 (73.99 MB)

 Trainable params: 19,395,652 (73.99 MB)

 Non-trainable params: 0 (0.00 B)

## 2.3 Vérification de l'architecture avec plot_model()

In [4]:
# Visualiser l'architecture
try:
    plot_model(model, to_file=os.path.join(MODELS_DIR, 'model_architecture.png'),
               show_shapes=True, show_layer_names=True, dpi=96)
    from IPython.display import Image
    Image(os.path.join(MODELS_DIR, 'model_architecture.png'))
except Exception as e:
    print('plot_model nécessite pydot/graphviz. Erreur:', e)

You must install graphviz (see instructions at https://graphviz.gitlab.io/download/) for `plot_model` to work.


## 2.4 Compilation, hyperparamètres et temps d'entraînement

In [8]:
# Compilation
LEARNING_RATE = 0.001
BATCH_SIZE = 32
EPOCHS = 15

model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print('lr:', LEARNING_RATE, '| batch:', BATCH_SIZE, '| epochs:', EPOCHS)

lr: 0.001 | batch: 32 | epochs: 15


In [10]:
checkpoint_path = os.path.join(MODELS_DIR, 'best_model.keras')

start = time.time()
history = model.fit(
    X_train,
    y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.1,
    class_weight=class_weight_dict,
    verbose=1
)
end = time.time()
print(f'\nDurée d\'entraînement: {(end-start):.2f}s ({(end-start)/60:.2f} min)')

Epoch 1/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 314s 2s/step - accuracy: 0.7571 - loss: 0.6460 - val_accuracy: 0.8737 - val_loss: 0.3606
Epoch 2/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 315s 2s/step - accuracy: 0.8801 - loss: 0.3449 - val_accuracy: 0.8826 - val_loss: 0.2679
Epoch 3/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 299s 2s/step - accuracy: 0.9153 - loss: 0.2375 - val_accuracy: 0.9164 - val_loss: 0.2251
Epoch 4/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 276s 2s/step - accuracy: 0.9411 - loss: 0.1711 - val_accuracy: 0.9306 - val_loss: 0.2064
Epoch 5/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 279s 2s/step - accuracy: 0.9624 - loss: 0.1207 - val_accuracy: 0.9573 - val_loss: 0.1669
Epoch 6/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 275s 2s/step - accuracy: 0.9697 - loss: 0.0907 - val_accuracy: 0.9270 - val_loss: 0.2045
Epoch 7/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 283s 2s/step - accuracy: 0.9753 - loss: 0.0705 - val_accuracy: 0.9502 - val_loss: 0.2029
Epoch 8/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 271s 2s/step - accuracy: 0.9747 - loss: 0.0688 - val_accu

## 2.5 Évaluation sur l'ensemble de test

In [ ]:
# Charger meilleur modèle si disponible
best_path = os.path.join(MODELS_DIR, 'best_model.keras')
if os.path.exists(best_path):
    model_eval = load_model(best_path)
    print('Chargé:', best_path)
else:
    model_eval = model
    print('Utilise modèle courant')

test_loss, test_acc = model_eval.evaluate(X_test, y_test, verbose=0)
print(f'Accuracy test: {test_acc:.4f} | Loss test: {test_loss:.4f}')

y_pred = model_eval.predict(X_test, verbose=0)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

## 2.6 Courbes d'apprentissage (accuracy / loss)

In [ ]:
acc = history.history.get('accuracy', history.history.get('acc', []))
val_acc = history.history.get('val_accuracy', history.history.get('val_acc', []))
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Train Acc')
plt.plot(epochs_range, val_acc, label='Val Acc')
plt.title('Accuracy')
plt.xlabel('Époques')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Train Loss')
plt.plot(epochs_range, val_loss, label='Val Loss')
plt.title('Loss')
plt.xlabel('Époques')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.show()

## 2.7 Matrice de confusion et rapport de classification

In [ ]:
cm = confusion_matrix(y_true_classes, y_pred_classes)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CATEGORIES, yticklabels=CATEGORIES)
plt.title('Matrice de confusion')
plt.xlabel('Prédit')
plt.ylabel('Réel')
plt.tight_layout()
plt.show()

print('\nRapport de classification:')
print(classification_report(y_true_classes, y_pred_classes, target_names=CATEGORIES, digits=4))

## 2.8 Exemples de prédictions correctes et incorrectes

In [ ]:
# Correctes
correct_idx = np.where(y_pred_classes == y_true_classes)[0]
plt.figure(figsize=(12, 4))
for i, idx in enumerate(correct_idx[:5]):
    plt.subplot(1, 5, i+1)
    img = (X_test[idx] * 255).astype(np.uint8)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.imshow(img)
    plt.title(f'Vrai:{CATEGORIES[y_true_classes[idx]]}\nPred:{CATEGORIES[y_pred_classes[idx]]}')
    plt.axis('off')
plt.suptitle('Prédictions correctes')
plt.tight_layout()
plt.show()

# Incorrectes
wrong_idx = np.where(y_pred_classes != y_true_classes)[0]
plt.figure(figsize=(12, 4))
for i, idx in enumerate(wrong_idx[:5]):
    plt.subplot(1, 5, i+1)
    img = (X_test[idx] * 255).astype(np.uint8)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.imshow(img)
    plt.title(f'Vrai:{CATEGORIES[y_true_classes[idx]]}\nPred:{CATEGORIES[y_pred_classes[idx]]}')
    plt.axis('off')
plt.suptitle('Prédictions incorrectes')
plt.tight_layout()
plt.show()

## 2.9 Sauvegarde du modèle et fonction predict_image()

In [ ]:
# Sauvegarder le meilleur modèle
best_path = os.path.join(MODELS_DIR, 'best_model.keras')
h5_path = os.path.join(MODELS_DIR, 'brain_mri_cnn.h5')
final_path = os.path.join(MODELS_DIR, 'brain_mri_cnn.keras')

if os.path.exists(best_path):
    model_eval.save(h5_path)
    model_eval.save(final_path)
    print('Sauvegardé:', h5_path, final_path)
else:
    model.save(h5_path)
    model.save(final_path)
    print('Sauvegardé depuis modèle courant')

In [ ]:
# Fonction predict_image(path)
def predict_image(image_path, target_size=(224,224), model_path=None):
    """Prédit la classe d'une image IRM cérébrale."""
    if model_path is None:
        model_path = os.path.join(MODELS_DIR, 'best_model.keras')
        if not os.path.exists(model_path):
            model_path = os.path.join(MODELS_DIR, 'brain_mri_cnn.keras')
    mdl = load_model(model_path)
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f'Image non trouvée: {image_path}')
    img = cv2.resize(img, target_size)
    img = img / 255.0
    img = np.expand_dims(img, axis=0)
    preds = mdl.predict(img, verbose=0)[0]
    pred_class = CATEGORIES[np.argmax(preds)]
    prob = float(preds[np.argmax(preds)])
    return pred_class, prob, preds

# Exemple d'usage (à adapter au chemin)
print('predict_image défini.')

## 2.10 Interface Streamlit (src/app.py)


In [ ]:
app_path = os.path.join(SRC_DIR, 'app.py')
print('Chemin:', app_path)
with open(app_path, encoding='utf-8') as f:
    content = f.read()
print('Taille:', len(content), 'caractères')